# Demo — Orquestador multi-agente BA Explore AI

Muestra la **delegación**: el Supervisor decide quién interviene (investigador → analista), valida con una rúbrica y recién entonces cierra. Requiere `GOOGLE_API_KEY` en `.env`.

Cada celda se ejecuta desde la carpeta del proyecto.

## 1. Topología del grafo

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from graph import construir_grafo
grafo = construir_grafo()
print(grafo.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	intake(intake)
	supervisor(supervisor)
	investigador(investigador)
	analista(analista)
	sintetizador(sintetizador)
	validador(validador)
	__end__([<p>__end__</p>]):::last
	__start__ --> intake;
	analista --> supervisor;
	intake --> supervisor;
	investigador --> supervisor;
	sintetizador --> validador;
	supervisor -.-> analista;
	supervisor -.-> investigador;
	supervisor -.-> sintetizador;
	validador --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Consulta que exige dos dominios: investigar (OSM, clima) y analizar (ranking, recorrido)

In [2]:
from langchain_core.messages import HumanMessage
from main import _mostrar

consulta = 'Armame un plan en Recoleta: museos y parrillas, a menos de 2 km. Quiero saber si conviene salir por el clima.'
estado, contribuciones, decisiones = {}, [], []
async for evento in grafo.astream({'messages': [HumanMessage(content=consulta)]},
                                  config={'recursion_limit': 40}, stream_mode='updates'):
    for nodo, upd in evento.items():
        if upd:
            _mostrar(nodo, upd)
            contribuciones += upd.get('contribuciones', [])
            decisiones += upd.get('decisiones', [])
            estado.update({k: v for k, v in upd.items() if k not in ('messages', 'contribuciones', 'decisiones')})

🧭 INTAKE (reglas): categorías=['turismo', 'gastronomia'] ubicación='recoleta' radio=2000 m



🧠 SUPERVISOR — decisión 1: No hay lugares recuperados ni datos de clima aún. El investigador debe buscar museos (turismo), parrillas (gastronomia) en Recoleta con radio 2000 y consultar el clima.
   → destino: investigador
   📝 instrucción: Buscá museos y parrillas en Recoleta en un radio de 2000 metros utilizando OpenStreetMap, y consultá el clima actual en Buenos Aires.


🔎 INVESTIGADOR usó ['buscar_lugares', 'buscar_lugares', 'pronostico_clima']
   Lugares por categoría: {'turismo': 25, 'gastronomia': 25}. Recuperé 25 lugares para la categoría turismo y 25 lugares para la categoría gastronomia utilizando OpenStreetMap, además del pronóstico del clima de Buenos Aires. Ninguna herramienta falló.



🧠 SUPERVISOR — decisión 2: El investigador ya obtuvo 25 lugares para turismo y 25 para gastronomía, además del clima. Falta que el analista procese los datos, genere los rankings y arme el recorrido.
   → destino: analista
   📝 instrucción: Procesa los datos recuperados: genera el ranking híbrido para turismo y gastronomía, arma un recorrido óptimo y evalúa el clima para salir.


C:\Users\MatiPC\OneDrive\Desktop\IA ENGINEERRING\Entrega 6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (2 outputs left out)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6215.03it/s]

📊 ANALISTA usó ['rankear_lugares', 'rankear_lugares', 'calcular_recorrido', 'evaluar_clima_plan']
   Rankings: ['turismo', 'gastronomia']; recorrido: sí. El ranking híbrido seleccionó en turismo al Paseo de la Recoleta y el Cementerio de la Recoleta, y en gastronomía a Pasta Recoleta Bodegon y Urban. 
El recorrido óptimo conectado suma una distancia total de 0,36 km entre las 4 paradas elegidas. 
La evaluación del clima indica que el plan es apto para realizar actividades al aire libre con una probabilidad de lluvia máxima del 39%.



🧠 SUPERVISOR — decisión 3: El investigador ha recuperado exitosamente las categorías requeridas y el analista ya ha generado los rankings, el recorrido y la evaluación del clima. Se cumplen todos los criterios de cobertura, análisis y suficiencia.
   → destino: FINISH



✍️  SINTETIZADOR: redactó el itinerario (solo con ids de la evidencia)
✅ VALIDADOR: {'ok': True, 'ids_invalidos_descartados': [], 'uso_fallback': False, 'categorias_sin_resultados': [], 'categorias_sin_ranking': [], 'paradas': 4}


## 3. Respuesta final (validada contra la evidencia)

In [3]:
from IPython.display import Markdown
Markdown(estado['respuesta'])

### Itinerario cerca de Recoleta
Acá tenés tu plan en Recoleta. No se encontraron datos de la categoría museos, pero te armamos el recorrido con atracciones y gastronomía.
1. **Paseo de la Recoleta** (attraction, turismo) · 0.056 km del centro · [OSM](https://www.openstreetmap.org/node/5405017227) — Es una atracción imperdible en la zona de Recoleta.
2. **Pasta Recoleta Bodegon** (restaurant, gastronomia) · 0.057 km del centro · [OSM](https://www.openstreetmap.org/node/3205426280) — Un restaurante ideal para comer algo rico cerca.
3. **urban** (restaurant, gastronomia) · 0.076 km del centro · [OSM](https://www.openstreetmap.org/node/3205426301) — Otro restaurante disponible para disfrutar en el recorrido.
4. **Cementerio de la Recoleta** (attraction, turismo) · 0.117 km del centro · [OSM](https://www.openstreetmap.org/way/183842128) — Una atracción destacada ubicada a pocos metros.

**Recorrido sugerido:** ~0.36 km en total (en línea recta, no tiempo de viaje).
**Clima:** El clima está apto para actividades al aire libre.

*Datos de OpenStreetMap. Horarios, precios y disponibilidad no están verificados.*

## 4. Quién aportó qué (estado compartido estructurado)

In [4]:
# 'contribuciones' y 'decisiones' se acumulan en el estado compartido con reducers (operator.add)
print('DECISIONES DEL SUPERVISOR')
for d in decisiones:
    print(f"  paso {d['paso']}: LLM propuso '{d['propuesta_llm']}' -> destino '{d['destino']}'" + (f" (override: {d['override']})" if d['override'] else ''))
print('\nCONTRIBUCIONES POR AGENTE')
for c in contribuciones:
    print(f"  [{c['agente']}] herramientas={c['herramientas']}")

DECISIONES DEL SUPERVISOR
  paso 1: LLM propuso 'investigador' -> destino 'investigador'
  paso 2: LLM propuso 'analista' -> destino 'analista'
  paso 3: LLM propuso 'FINISH' -> destino 'FINISH'

CONTRIBUCIONES POR AGENTE
  [investigador] herramientas=['buscar_lugares', 'buscar_lugares', 'pronostico_clima']
  [analista] herramientas=['rankear_lugares', 'rankear_lugares', 'calcular_recorrido', 'evaluar_clima_plan']
